<a href="https://colab.research.google.com/github/williamli218/mais202-team-2-fraud-bursts/blob/main/notebooks/workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1.1 Setup

Run the code below in order to downlaod your repo onto Colab (using your GitHub token). It will then update it with git pull, load your Kaggle keys, install the needed packages, and make src/ and config.py importable. Run it once at the start of every session.

In [ ]:
REPO_URL = "https://github.com/williamli218/mais202-team-2-fraud-bursts"

import os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if "<your-team>" in REPO_URL:
        raise ValueError("Edit REPO_URL at the top of this cell to your team's repository first.")
    from google.colab import userdata

    def _secret(name):
        try:
            return userdata.get(name)
        except Exception as e:
            raise RuntimeError(
                f"Colab Secret {name} is missing or not shared with this notebook. "
                "Open the key icon in the left sidebar, add it, and switch on Notebook access."
            ) from e

    token = _secret("GITHUB_TOKEN")
    for name in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        os.environ[name] = _secret(name)

    repo = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.isdir(repo):
        auth_url = REPO_URL.replace("https://", f"https://{token}@")
        subprocess.run(["git", "clone", "--quiet", auth_url], check=True)
    os.chdir(repo)
    subprocess.run(["git", "pull", "--quiet"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # Not in Colab: assume this notebook is open from inside the repository.
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

sys.path.insert(0, os.getcwd())
import config
print("repository:", os.getcwd(), "| template version:", config.TEMPLATE_VERSION)
print("TRACK in config.py:", config.TRACK, "" if config.TRACK else "<- one teammate sets this and pushes config.py")

The following code tells git who you are (name and email) so your commits are labelled "William Li" on GitHub. The noreply email keeps your real one private. Run it once per session, before committing

In [ ]:
!git config user.name "William Li"
!git config user.email "williamli218@users.noreply.github.com"

## 1.2 Download and load data

In [ ]:
from src import data

pool, test = data.load_track02()

## 1.3 Quick look

In [ ]:
data.quick_look(pool, "is_fraud", "train pool")
data.quick_look(test, "is_fraud", "test")
attacked = pool.groupby("cc_num", observed=True)["is_fraud"].max()
print(f"cards in pool: {len(attacked):,}, attacked (at least one fraud): {int(attacked.sum()):,}")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))

counts = train["is_fraud"].value_counts().sort_index()
ax1.bar(["Legitimate", "Fraud"], counts.values, color=["#4C72B0", "#C44E52"])
ax1.set_yscale("log")
ax1.set_ylabel("Transactions (log scale)")
ax1.set_title(f"Class balance: {100 * train['is_fraud'].mean():.2f}% fraud")

hour = pd.to_datetime(train["trans_date_trans_time"]).dt.hour
by_hour = pd.crosstab(hour, train["is_fraud"], normalize="columns") * 100
ax2.plot(by_hour.index, by_hour[0], label="Legitimate")
ax2.plot(by_hour.index, by_hour[1], label="Fraud", color="#C44E52")
ax2.set_xlabel("Hour of day")
ax2.set_ylabel("% of that class")
ax2.set_title("When transactions happen")
ax2.legend()

plt.tight_layout()
plt.show()

## 1.4 Fixed test split

In [ ]:
print("train pool:", pool["trans_date_trans_time"].min(), "->", pool["trans_date_trans_time"].max())
print("test:      ", test["trans_date_trans_time"].min(), "->", test["trans_date_trans_time"].max())

## 1.5 Validation split
In order to choose for a validation cut date, a month-by-month view of transactions and fraud is displayed in the following table.

In [ ]:
import pandas as pd
t = pd.to_datetime(pool["trans_date_trans_time"])
month = t.dt.to_period("M")

fraud = pool[pool["is_fraud"] == 1]
first_fraud = fraud.assign(t=t[fraud.index]).groupby("cc_num", observed=True)["t"].min()

monthly = pd.DataFrame({
    "rows": pool.groupby(month).size(),
    "fraud_tx": pool.groupby(month)["is_fraud"].sum(),
    "attacks_starting": first_fraud.dt.to_period("M").value_counts(),
}).fillna(0).astype(int)
monthly["fraud_rate_%"] = (100 * monthly["fraud_tx"] / monthly["rows"]).round(3)
monthly

fr = fraud.assign(t=t[fraud.index]).groupby("cc_num", observed=True)["t"].agg(["min", "max"])

rows = []
for cut in pd.date_range("2019-09-01", "2020-04-01", freq="MS"):
    rows.append({
        "cut": cut.date(),
        "train_months": round((cut - t.min()).days / 30.4, 1),
        "valid_months": round((t.max() - cut).days / 30.4, 1),
        "attacks_in_valid": int((fr["min"] >= cut).sum()),
        "straddling_cut": int(((fr["min"] < cut) & (fr["max"] >= cut)).sum()),
    })
pd.DataFrame(rows)

Choice of validation cut date: January 1st, 2020.

Done with helper functions from src/split.py

Reasons:
1. It gives a full year of training (all of 2019, every season, including holidays)
2. It gives a validation window close to the test set's length  (5.7 vs 6.3 months)
3. It gives plenty of attacks in validation (235, enough for a stable median)
4. Almost no attacks are split by the cut (only 2)

In [ ]:
from src import split

try:
    train, valid = split.make_validation_split(pool)
except NotImplementedError as e:
    train = valid = None
    print(e)
else:
    split.check_split(train, valid, *split.default_columns())
    print(f"train: {len(train):,} rows, valid: {len(valid):,} rows; both checks passed")

In [ ]:
!python -m pytest -q tests/test_leakage.py

## 1.6 Features
Helper function from src/features.py


In [ ]:
from src import features

features.build_features(pool.head(10))

## 1.7 The trivial baseline
The baseline calls every row legitimate: every score is 0. It's here to be scored with your metric and with accuracy, side by side, before you train anything. On this track it will look very accurate and catch nothing. That gap is the point of the evaluation metric section of your proposal. Helper function from src/metrics.py

In [ ]:
from src.models import always_legitimate
from sklearn.metrics import accuracy_score

baseline_scores = always_legitimate(len(pool))
print("baseline scores:", baseline_scores[:5], "... (all zeros)")
labelled = pool["is_fraud"] != -1
print("accuracy on the train pool if we predict 'legitimate' everywhere:",
      round(accuracy_score(pool["is_fraud"][labelled], (baseline_scores[labelled] >= 0.5).astype(int)), 4))

from src import metrics
y_valid = valid["is_fraud"].to_numpy()
s_valid = always_legitimate(len(valid))
print("headline metric:", metrics.median_frauds_before_hold(y_valid, s_valid, valid["cc_num"].to_numpy(), max_legit_hold_rate=0.01))
print("accuracy:       ", accuracy_score(y_valid, (s_valid >= 0.5).astype(int)))

## 1.8 Sequences
`make_sequences` turns the table into one fixed-length window per
transaction: the card's last `n` transactions ending at it, oldest first,
zero-padded while the card has fewer than `n`. `X` is what a sequence model
reads; `y` is the label of the last transaction in each window; `meta` keeps
the card, the time and how many rows in the window are real.

In [ ]:
X, y, meta = data.make_sequences(pool, n=10, feature_cols=["amt"])
print("X:", X.shape, X.dtype, "| y:", y.shape, "| meta columns:", list(meta.columns))
print("memory for X: %.0f MB" % (X.nbytes / 1e6))
i = int(meta.index[meta["n_valid"] == 10][0])
print("one window (amount only), oldest first:", X[i, :, 0].round(2))

In [ ]:
!python -m pytest -q tests/test_metrics.py tests/test_leakage.py

## 2.1 Simple baseline model: logistic regression
Train model and evaluate score

In [ ]:
from src import features

feats = features.build_features(pool)
X_train, X_valid = feats.loc[train.index], feats.loc[valid.index]
model = models.train_logistic_regression(X_train, train["is_fraud"])
s_valid = model.predict_proba(X_valid)[:, 1]
print("headline metric:", metrics.median_frauds_before_hold(
    valid["is_fraud"], s_valid, valid["cc_num"], max_legit_hold_rate=0.01))